# Book-to-Price (BP) — Risk-Factor Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for building the
**BP** style factor exactly as MSCI describes it in *Barra USE4 Empirical Notes*
(Appendix A, p. 52). It is **not** a research notebook. The deliverable is a
clean, USE4-faithful Book-to-Price factor written to parquet, suitable for
input to a multi-factor risk model.

**Audience.** You — plus whatever AI assistant you are driving. Treat its output as deeply untrustworthy until audited. Follow the stages linearly. Each stage has:
- ✅ **PDF SPEC** — exact verbatim quote or paraphrase from the USE4 documentation
- ❓ **NOT IN PDF** — something the PDF does not specify; a judgment call you must make
- 💡 **DEFAULT** — a reasonable default for any ❓ item, chosen to match standard Barra practice
- 🧪 **VALIDATE** — sanity checks to run after each stage

**Inputs:**
- `SHARADAR_SF1.csv` — Sharadar SF1 fundamental data (dimension=ARQ, PIT via `datekey`)
- `data/out/estu_monthly.parquet` — Estimation universe per signal_date (`in_estu`, `mcap`, monthly signal dates)

**Deliverable:** A parquet file `bp_use4.parquet` keyed by `(permaticker, signal_date)`
containing the standardized BP exposure for every stock in the coverage universe on every
monthly signal date.

**Companion specs:**
- `02_style_factors/eyld/eyld_spec.ipynb` — Earnings Yield (EYLD): complementary
  value factor; BP captures asset-based value, EYLD captures earnings-based value (§3.4, p. 16)

## §1. The USE4 BP spec — verbatim quotes

### 1a. Barra USE4 Empirical Notes, Appendix A (p. 52) — formal descriptor definition

> **Book-to-Price (BP)**
>
> *Definition:* `1.0 · BTOP`
>
> *BTOP — Book-to-price ratio:* "Last reported book value of common equity divided
> by current market capitalization."

### 1b. Barra USE4 Empirical Notes, Section 3.4 (p. 16) — economic role and EYLD complement

> "The Book-to-Price factor captures return differences based on a company's book
> value relative to its market price. It is considered complementary to the Earnings
> Yield factor: BP captures asset-based value while EYLD captures earnings-based value."

### 1c. Barra USE4 Methodology Notes, Section 2.3 (p. 9) — standardization rule

> *"Descriptors are standardized to have a mean of 0 and a standard deviation of 1.
> ... μ_l is the cap-weighted mean of the descriptor (within the estimation universe),
> and σ_l is the equal-weighted standard deviation."*

---

**That is all the PDFs say about BP construction specifically.** The following are
NOT covered:
- Which data vendor provides the book value
- Exact definition of "common equity" (total equity vs. total equity minus preferred)
- Whether to use the most recent quarterly or annual filing
- PIT alignment (filing date vs. fiscal period end)
- Treatment of negative book equity
- Staleness cutoff for fundamental reports

## §2. End-to-end pipeline

```
┌─────────────────────────────────────────────────────────────────────┐
│  STAGE 0:  Setup, imports, paths                                    │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 1:  Load Sharadar SF1 fundamentals (ARQ, PIT via datekey)   │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 2:  Build ESTU per signal_date                               │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 3:  (Skipped — no market benchmark needed for BP)           │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 4:  BP estimator — BTOP = equity / mcap                      │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 5:  Outlier treatment  (3σ default)                         │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 6:  Standardize  (cap-weighted mean=0, EW std=1)            │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 7:  Save deliverable                                         │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 8:  Validate                                                 │
└─────────────────────────────────────────────────────────────────────┘
```

**PIT discipline:** For any signal_date `t`, the only permissible data is records
with `datekey ≤ t` (Sharadar filing date). `calendardate` (fiscal period end) lags
behind `datekey` — using it instead would introduce look-ahead bias.

**No daily price data is used.** BP is purely fundamental (book value / market cap).
Market cap on signal_date comes from `estu_monthly.parquet`.

**Book value is a balance-sheet stock, not a flow.** Unlike EYLD (which uses TTM-summed
earnings), BP needs only the single most recent quarterly balance sheet. This is why
SF1 dimension ARQ is used rather than TTM.

## §3. Output schema — what the worker delivers

**Raw descriptor column:** `btop`
**Standardized score column:** `btop_score`

**File:** `data/out/bp_use4.parquet`

**Compression:** zstd, statistics=True

**Schema:**

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `signal_date` | Date | End-of-month rebalance date (last trading day of month) |
| `in_estu` | Bool | Whether this stock was in the estimation universe on this date |
| `mcap` | Float64 | Market cap on signal_date (from estu_monthly; used for cap-weighting) |
| `btop` | Float64 | **Raw descriptor** = equity / mcap (Stage 4 output) |
| `btop_score` | Float64 | **Final BP exposure** — standardized cross-sectionally (the deliverable) |
| `n_obs` | UInt32 | 1 if a valid ARQ fundamental observation was found within the lookback window; 0 otherwise |

**Coverage rules:**
- Compute `btop` and `btop_score` for **every stock with a valid ARQ observation**
  within the `LOOKBACK_DAYS` window, not just ESTU members.
- Standardization moments (mean, std) are computed using **ESTU members only**.
- Non-ESTU stocks get the *same* standardization parameters applied so the final
  factor is comparable across the coverage universe.

**Note on n_obs:** For this factor, `n_obs` is binary (1 = has a valid report within
the lookback window, 0 = stale or missing). It does not count historical observations
as it would for a rolling-window time-series factor.

## §4. STAGE 0 — Setup, imports, paths

Standard imports. Use polars, not pandas, throughout (project convention).

✅ **PDF SPEC — BP parameters:**

> `BP = 1.0 · BTOP` — *"Book-to-price ratio: last reported book value of common equity
> divided by current market capitalization."* (Appendix A, p. 52)

❓ **NOT IN PDF — staleness cutoff (item 5):** The PDF does not specify when a
fundamental report is considered too old to use.

💡 **DEFAULT:** Discard any report with `datekey` more than 18 months (~548 calendar
days) before `signal_date`. Consistent with EYLD precedent — book value changes even
more slowly than earnings, so this is a conservative upper bound.

❓ **NOT IN PDF — minimum observations threshold:** For a spot balance-sheet measure
with no rolling window, any single valid observation within the lookback window suffices.

💡 **DEFAULT:** `MIN_OBS = 1`

```python
# ───── Parameters ─────────────────────────────────────────────────────────────
# BP = 1.0 × BTOP (Appendix A, p. 52) — single descriptor, weight 1.0 is implicit;
# no weight constant is defined (it would be dead code in the build).
LOOKBACK_DAYS   = 548    # 💡 DEFAULT (NOT IN PDF) — ~18 months max filing staleness
MIN_OBS         = 1      # 💡 DEFAULT (NOT IN PDF) — binary: has valid report or not

# 💡 DEFAULT (NOT IN PDF) — calendar start
CALENDAR_START  = date(1999, 1, 1)

# -- Factor type (read by /build-factor to select boilerplate template)
FACTOR_TYPE = "fundamental"   # fundamental (not time_series)
```

## §5. STAGE 1 — Load Sharadar SF1 fundamentals

✅ **PDF SPEC:** BTOP uses "last reported book value of common equity":
> "Last reported book value of common equity divided by current market capitalization."
>
> (Appendix A, p. 52)

❓ **NOT IN PDF — common equity definition (item 1):** USE4 specifies "book value of
common equity" (= total equity minus preferred equity). Sharadar SF1 has no separate
balance-sheet field for preferred equity or preferred stock; the only equity-related
balance-sheet field is `equity` (total stockholders' equity).

💡 **DEFAULT:** Use `equity` directly as BE. For the vast majority of U.S. companies
this is equivalent to common equity, since most firms have no preferred stock. Firms
with meaningful preferred stock are rare in the ESTU; the effect on the cross-section
is immaterial.

❓ **NOT IN PDF — SF1 dimension (item 2):** SF1 has multiple dimension codes.
Book value is a balance-sheet stock (a snapshot), not a flow — it should not be TTM-summed.

💡 **DEFAULT:** Use `dimension = "ARQ"` (as-reported quarterly). This gives the most
recent quarterly balance sheet snapshot per filing. Do NOT use `"TTM"` (correct for
income-statement flows like earnings) — summing four quarters of equity would double-count.

❓ **NOT IN PDF — PIT alignment (item 3):** Sharadar SF1 has two date columns:
- `calendardate` — the fiscal period end date (e.g., 2024-12-31)
- `datekey` — the date the data was released / made available (the filing date)

Using `calendardate` introduces look-ahead bias of ~45–90 days (the typical earnings
announcement lag). `datekey` is the correct PIT date.

💡 **DEFAULT:** Filter SF1 to only rows with `datekey ≤ signal_date` when selecting
the most recent fundamental observation per stock per signal_date.

🧪 **VALIDATE:**
- `dimension == "ARQ"` rows only in the loaded table
- `equity` is non-null for the vast majority of rows (>90%)
- `datekey` ≤ `calendardate` + 120 days (filing within 4 months of period end — flag outliers)
- Date range covers full history from CALENDAR_START

## §6. STAGE 2 — Build ESTU per signal_date

**Identical to the shared USE4 build infrastructure** — no BP-specific decisions here.

✅ **PDF SPEC:** The estimation universe (ESTU) is used to compute standardization
moments. Non-ESTU stocks are standardized using the same moments.

💡 **DEFAULT:** Load `estu_monthly.parquet` — prebuilt by `estu_build.ipynb`. This file
provides `(permaticker, signal_date, in_estu, mcap)` for every stock on every
end-of-month signal date.

**Key: `mcap` from `estu_monthly` is the signal_date market cap used as the denominator
for BTOP.** Do not use the `marketcap` column from SF1, which is measured as of the
report filing date — not the signal date.

🧪 **VALIDATE:**
- ESTU has ~2,400–2,600 `in_estu = True` stocks per date (post-2005)
- Signal dates are the last trading day of each month
- `mcap` is non-null and positive for all `in_estu = True` rows

## §7. STAGE 3 — Market benchmark

**Not required for BP.** This factor is computed from a fundamental accounting ratio;
no market return data is needed. Skip this stage entirely.

(Stage 3 computes a cap-weighted ESTU market return series, which is used by time-series
factors such as Beta, Momentum, NLB, and NLS to compute excess returns. BP does not
involve any return time series.)

## §8. STAGE 4 — BP estimator

✅ **PDF SPEC (Barra USE4 Empirical Notes, Appendix A, p. 52):**

> **BTOP — Book-to-price ratio:**
>
> *"Last reported book value of common equity divided by current market capitalization."*
>
> `BP = 1.0 · BTOP`

❓ **NOT IN PDF — common equity vs. total equity (item 1):** USE4 says "common equity"
(= total equity − preferred equity), but Sharadar has no separate PREFEQUITY field.

💡 **DEFAULT:** Use `equity` as BE directly. Equivalent to common equity for most U.S.
firms; preferred stock is rare in the ESTU and immaterial to the cross-section.

❓ **NOT IN PDF — mcap denominator source (item 4):** "Current market capitalization"
must be the market cap on signal_date, not the market cap on the report filing date.

💡 **DEFAULT:** Use `mcap` from `estu_monthly` (signal_date market cap) as ME for all
stocks. Do NOT use SF1 `marketcap`, which is measured as of `datekey` and may be 1–4
months stale.

❓ **NOT IN PDF — negative book equity (item 6):** The USE4 spec does not say whether
negative equity (more liabilities than assets — distressed or highly-leveraged firms)
should be treated as missing.

💡 **DEFAULT:** Allow negative `btop`. Firms with negative book equity are genuinely
informative value signals — they are distressed or highly-levered and should rank at
the bottom of the cross-section (low book-to-price). The 3σ outlier trim in Stage 5
handles extreme outliers. Do NOT set negative equity to null.

❓ **NOT IN PDF — "most recent" report selection (item 3):** For each stock on each
signal_date, multiple ARQ rows may be available (one per quarter). We need exactly one.

💡 **DEFAULT:** Select the row with the maximum `datekey ≤ signal_date`, subject to
`datekey > signal_date − LOOKBACK_DAYS`. This is the most recently filed quarterly
balance sheet available PIT. If no row exists within the lookback window, the stock
gets `btop = null`.

---
*The section above (PDF SPEC quote through final 💡 DEFAULT) is the `STAGE4_DESCRIPTION`
that `/build-factor` will inject verbatim into the Stage 4 stub docstring. Content below
this line is supplementary guidance for the implementer and is not extracted.*
---

**Implementation notes:**
- Sort `sf1_arq` by `(permaticker, datekey)` and use `join_asof` with `strategy="backward"`
  on `datekey` to efficiently find the most recent report per stock per signal_date.
- After the join, apply the staleness filter: drop rows where
  `signal_date − datekey > LOOKBACK_DAYS` calendar days.
- Compute the descriptor:

```
btop_i = equity_i / mcap_i
```

- Guard: filter `mcap > 0` before division (rare Sharadar data errors); mark `btop = null`
  for missing `equity`.
- `n_obs = 1` for every stock that passes the lookback filter.

🧪 **VALIDATE:**
- Raw `btop` cross-sectional range: most stocks in roughly −1.0 to +3.0
  (book value is typically 20–200% of market cap for healthy firms)
- Median `btop` (ESTU) should be modestly positive (~0.3–0.6 for typical U.S. equity market)
- High book-to-price value stocks (financials, mature industrials, energy) → highest `btop`
- High-multiple growth stocks (technology, biotech) → lowest (near-zero) `btop`
- Distressed firms with negative equity → most negative `btop`
- Coverage ≥ 4,000 stocks per date post-2005

## §9. STAGE 5 — Outlier treatment

❓ **NOT IN PDF for BP specifically.** Methodology Notes Section 2.2 (p. 8)
describes a general outlier-treatment algorithm that applies to all descriptors:

> *"We trim these observations to three standard deviations from the mean."*

💡 **DEFAULT:** Apply 3σ trimming per signal_date, computed within ESTU using
cap-weighted mean ± 3 × equal-weighted std. Applied to all stocks in the coverage
universe.

**Note on negative BTOP:** Negative `btop` values must NOT be removed before trimming —
they are valid observations. The lower 3σ bound will naturally accommodate the negative
tail from distressed firms. The upper 3σ bound clips high book-to-price micro-caps
(e.g., failed financials trading below tangible book).

🧪 **VALIDATE:**
- ~0.5–2% of ESTU rows should be clipped at either bound per date
- After trimming, cross-sectional skewness and kurtosis of `btop` should be lower
- Upper-tail clips concentrated in financials (banks, insurers with large balance sheets
  relative to market cap)

## §10. STAGE 6 — Standardize (cap-weighted mean=0, EW std=1)

✅ **PDF SPEC (Methodology Notes §2.3, p. 9):**

> *"μ_l is the cap-weighted mean of the descriptor (within the estimation universe),
> and σ_l is the equal-weighted standard deviation."*

**Per signal_date `t`, using only ESTU members:**

```
μ_t  = Σ_{i ∈ ESTU_t} (mcap_i · btop_i) / Σ mcap_i   # cap-weighted mean
σ_t  = std_{i ∈ ESTU_t}(btop_i)                         # equal-weighted std
btop_score_{i,t} = (btop_i − μ_t) / σ_t                # applied to ALL i
```

Three critical points:
1. Moments estimated on **ESTU only**; applied to **every stock** in the coverage universe.
2. Mean is **cap-weighted**; std is **equal-weighted**.
3. Cap-weighting the mean ensures the cap-weighted market portfolio has zero BP exposure.

🧪 **VALIDATE:**
- `Σ_{i ∈ ESTU} (mcap_i · btop_score_i) / Σ mcap_i ≈ 0` (cap-weighted mean = 0 by construction)
- `std_{i ∈ ESTU}(btop_score_i) ≈ 1` (equal-weighted std = 1 by construction)

## §11. STAGE 7 — Save deliverable

Write the final panel to `data/out/bp_use4.parquet`.
Compression: zstd. Statistics: True.

Include `btop` (raw) and `btop_score` (standardized) for downstream use.
The downstream consumer (factor model, audit notebook) will use `btop_score`.

## §12. STAGE 8 — Validation

### Required checks (all must pass before signing off)

**1. Standardization moments on ESTU:**
```
cap-weighted mean of btop_score ≈ 0   (|mean| < 1e-6)
equal-weighted std of btop_score ≈ 1   (|std − 1| < 0.02)
```

**2. Raw descriptor sanity:**
```
Median btop (ESTU) in range [0.20, 0.80]  (typical U.S. market book-to-price range)
Negative btop values present (distressed firms) — flag if none found post-2000
Max btop post-trim < 15.0                  (guards against unit mismatch; the 3σ upper bound
                                            legitimately reaches ~11 on wide-dispersion dates
                                            — BTOP is right-skewed)
```

**3. Coverage stability:**
```
≥ 4,000 stocks with non-null btop_score per date post-2005
```

**4. Factor stability (month-over-month rank correlation):**
```
MoM Spearman ρ > 0.97  (book value updates only ~4×/year; BP is highly persistent)
```

**5. Economic direction:**
```
Q5 (high BTOP) contains deep-value stocks with high book relative to price.
Q1 (low BTOP) contains high-multiple growth stocks or firms with negative equity.
Spot check: financials (banks, insurers) and mature energy → Q5.
Spot check: high-multiple technology stocks → Q1.
```

**6. Disk vs memory equivalence:**
```
max abs diff of btop values after read-back < 1e-10
```

---

**Shared validation conventions (all factors, 2026-06-10):**
- **Coverage (check 3)** is evaluated on **completed months only** — the final
  signal date is the in-progress month (freshest data can lag the Sharadar
  refresh) and is excluded from the floor.
- **Fraction of scores in [−3, +3]** is reported for the full universe *and* for
  ESTU; the ESTU figure is the operationally relevant one (non-ESTU micro-caps
  pull the all-universe tail).
- Common check machinery lives in `common/`, your shared utilities.

## §13. Master list of ❓ NOT-IN-PDF decisions

| # | Decision | Default chosen | Alternative | Where to revisit |
|---|---|---|---|---|
| 1 | Common equity definition | `equity` (no PREFEQUITY in Sharadar) | Proxy PREFEQUITY via `prefdivis × P/E` (complex, unreliable) | If a preferred equity balance-sheet field is added to data pipeline |
| 2 | SF1 dimension | ARQ (as-reported quarterly) — book value is a balance-sheet stock, not a flow | ARY (annual); loses interim periods | Never — ARQ is always correct for balance-sheet items |
| 3 | PIT alignment | `datekey` (filing date) as PIT constraint | `calendardate` + fixed lag (e.g. +60 days) | If Sharadar datekey quality is suspected |
| 4 | ME denominator source | `estu_monthly.mcap` on signal_date | SEP `marketcap` on signal_date (algebraically equivalent) | Never — estu_monthly mcap is the standard source for all factors |
| 5 | Staleness cutoff | 548 calendar days (~18 months) — same as EYLD | 365 days (tighter), 730 days (looser) | If stale book values are suspected to add noise |
| 6 | Negative BE treatment | Retain as valid negative BTOP | Mark as missing; exclude distressed firms | Only if factor is used in an application where negative book equity is meaningless |
| 7 | Zero/missing ME | Mark BTOP as null; exclude from computation | Floor ME at a small positive value | Never — zero or missing mcap is a data error, not a valid observation |
| 8 | Outlier trim multiplier | 3σ (PDF default — appropriate here; BTOP does not have the Gaussian-distribution anomaly of SIZE) | 4σ (only needed for SIZE due to cap-correlated distribution) | Stage 5 |
| 9 | MIN_OBS threshold | 1 (binary: has valid ARQ or not) | Higher if filing quality check required | If n_obs proves uninformative |
| 10 | Calendar start | 1999-01-01 | 2000-01-01 (avoid thin Sharadar pre-2000 coverage) | If early data quality is poor |

## §14. Common pitfalls — read this before coding

**Pitfall 1: Using calendardate instead of datekey for PIT**
Sharadar's `calendardate` is the end of the fiscal period (e.g., Dec 31). Book values
are not *reported* until 30–90 days later (`datekey`). Using `calendardate ≤ signal_date`
as the filter introduces 1–3 months of look-ahead bias on every signal date — a
systematic form of data snooping that would inflate backtest returns.

**Pitfall 2: Using TTM instead of ARQ for book value**
`equity` (total stockholders' equity) is a balance-sheet stock — a snapshot, not a flow.
Summing four quarters of equity (as TTM does) produces a meaningless number (~4× the
actual equity). Always use `dimension = "ARQ"` for balance-sheet items. Only income-
statement items (earnings, cash flow) should be TTM-annualized.

**Pitfall 3: Treating negative equity as missing**
Distressed firms, highly-leveraged buyouts, and firms with large accumulated deficits
legitimately have negative book equity. These firms belong at the bottom of the BP
cross-section (low book-to-price = expensive relative to book). Setting negative equity
to null would artificially inflate the factor's Q1 and destroy the value spread.

**Pitfall 4: Using SF1 `marketcap` instead of `estu_monthly.mcap` as the denominator**
SF1's `marketcap` field reflects the market cap as of the `datekey` (report filing date),
which may be 1–4 months before the signal date. For any price-ratio factor, the
denominator must be the market cap on the *signal date* (when the factor is evaluated).
Always use `estu_monthly.mcap` for the ME denominator.

**Pitfall 5: Not guarding against mcap = 0 or null**
Rare but real: some Sharadar records have `marketcap = 0` (data errors) or null.
Always filter `mcap > 0` from `estu_monthly` before computing BTOP. Without this guard,
division by zero produces inf values that corrupt the cross-sectional standardization.

## §15. Final summary — what "done" looks like

**The deliverable is one file:** `data/out/bp_use4.parquet`

**Acceptance criteria:**

1. ✅ Schema matches §3 (7 columns, exact dtypes)
2. ✅ All 6 required validation checks in §12 pass within tolerance
3. ✅ ESTU has ~2,400–2,600 names per date post-2005, stable over time
4. ✅ Cap-weighted mean of `btop_score` is machine-zero for every date in ESTU
5. ✅ Equal-weighted std of `btop_score` is 1.0 (to 2 decimal places) for every date in ESTU
6. ✅ Median raw `btop` (ESTU) is in [0.20, 0.80] for every post-2005 date
7. ✅ Coverage of `btop_score` across coverage universe ≥ 4,000 stocks per date post-2005
8. ✅ Month-over-month rank stability ρ > 0.97
9. ✅ All 10 ❓ NOT-IN-PDF decisions in §13 are documented in the build notebook code

**Once BP is built and passes validation, it is a standalone factor** — no other
USE4 factors depend on BP as an upstream input. It feeds directly into the
multi-factor risk model as one of the value style factors, alongside EYLD.